# 03 | Evaluation and findings

**My question:** can Scout find the right past incident and give an answer an investigator can check?

I tested **two different things**:
1. **Retrieval:** keyword overlap versus TF-IDF. **Hit@5** means the correct source ID is among the first five search results. I also measured search time.
2. **Raw Llama answers:** factual correctness, source citations, clear refusals when information is unavailable, and generation time.

**Baseline:** keyword overlap. **Provisional:** the source pages and model grades are not fully human-approved, and this question set was prepared retrospectively.

The scored results are saved in `reports/`, so this report is readable even without the original local CSV. The later checked-plan interface is **not** what the raw Llama figures evaluate.

## 1. Questions and expected sources

I have **12 factual questions** with expected published incident IDs, plus **three questions the public data cannot answer**. I reworded the factual questions rather than copy their titles.

I had already begun developing Scout before preparing the set. That does **not** meet the rubric's ideal of prospectively frozen questions, so I label every result provisional. The original vendor pages still need full human confirmation.

In [ ]:
# Locate the Scout project folder and load the 15 fixed evaluation questions.
# read_csv() reads a CSV file into a pandas DataFrame (table).
# The .eq("true") checks convert written true/false values into Booleans.

from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

local_questions = ROOT / "evals/incident_eval_questions.csv"
archived_questions = ROOT / "evals/incident_eval_questions_provisional.csv"
question_path = local_questions if local_questions.exists() else archived_questions

questions = pd.read_csv(question_path, keep_default_na=False)
questions["answerable"] = questions["answerable"].astype(str).str.lower().eq("true")
questions["reviewed_by_human"] = questions["reviewed_by_human"].astype(str).str.lower().eq("true")

print("Question set:", question_path.relative_to(ROOT))
print("Total questions:", len(questions))
print("Answerable questions:", int(questions["answerable"].sum()))
print("Questions requiring an explicit refusal:", int((~questions["answerable"]).sum()))
print("Independently reviewed and approved:", int(questions["reviewed_by_human"].sum()))
assert len(questions) >= 15 and (~questions["answerable"]).sum() >= 3
display(questions[["question_id", "question", "answerable", "reviewed_by_human"]])
if not questions["reviewed_by_human"].all():
    print("STATUS: PROVISIONAL — independent source review still outstanding.")

## 2. Retrieval against the keyword baseline

Both methods searched the same 12 factual questions and returned up to five incidents. **Keyword found 9/12; TF-IDF found 8/12.** TF-IDF was faster once the index was built (**0.007s vs 0.551s** mean query time). These are search-only times, not complete app responses.

![Recorded retrieval success chart](../reports/retrieval_hit_at_5.svg)

![Recorded search time chart](../reports/retrieval_query_latency.svg)

The code below recalculates the hit counts from the saved per-question results.

In [ ]:
# Compare the two search methods on the same twelve answerable questions.
# .mean() on True/False values calculates the share of successful hits.
# The timings come from the measured run and EXCLUDE building the search index.

observed = pd.read_csv(ROOT / "reports/retrieval_baseline_2026-11.csv")
for col in ["keyword_hit_at_5", "tfidf_hit_at_5"]:
    observed[col] = observed[col].astype(str).str.lower().eq("true")

summary = pd.DataFrame([
    {"approach": "Keyword overlap", "hits": int(observed["keyword_hit_at_5"].sum()),
     "questions": len(observed), "hit_at_5": observed["keyword_hit_at_5"].mean(),
     "mean_search_seconds": 0.551},
    {"approach": "TF-IDF", "hits": int(observed["tfidf_hit_at_5"].sum()),
     "questions": len(observed), "hit_at_5": observed["tfidf_hit_at_5"].mean(),
     "mean_search_seconds": 0.007},
])

display(summary.style.format({"hit_at_5": "{:.1%}", "mean_search_seconds": "{:.3f}"}))
assert len(observed) == 12
assert summary["hits"].tolist() == [9, 8]
print("Measured search time only; index construction excluded.")

In [ ]:
# Draw two separate charts: retrieval success, then query time.
# A bar chart compares named methods; values are labelled for easy reading.
# These graphs do NOT show the model's factual answer accuracy.

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(summary["approach"], summary["hit_at_5"] * 100)
ax.set_ylim(0, 100)
ax.set_ylabel("Expected incident in first five results (%)")
ax.set_title("Known-incident retrieval on 12 questions")
for x, value in enumerate(summary["hit_at_5"] * 100):
    ax.text(x, value + 2, f"{value:.1f}%", ha="center")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(summary["approach"], summary["mean_search_seconds"])
ax.set_ylabel("Mean seconds per query")
ax.set_title("Search latency after index construction")
for x, value in enumerate(summary["mean_search_seconds"]):
    ax.text(x, value + 0.015, f"{value:.3f}s", ha="center")
plt.tight_layout()
plt.show()

## 3. Where retrieval went wrong

I inspected the returned titles for five difficult questions:

- **Azure Q02:** keyword matching preferred other suppliers' reports about Azure.
- **Gamma Q05:** TF-IDF preferred other DNS incidents.
- **LangSmith Q07:** both missed `APAC` when the question said "Asia-Pacific".
- **Retell AI Q09:** TF-IDF ranked general traffic problems above the target call-latency report.
- **Vanta Q11:** both missed the specific "Chat with us" button phrasing.

**My conclusion:** similar words do not necessarily mean the same vendor, service or incident. Vendor-aware or semantic retrieval would be worth trying on **new questions**, not by quietly tuning against this test.

In [ ]:
# "~" means NOT and "|" means OR for pandas Boolean columns.
# This selects questions that EITHER retriever missed, for error analysis.

misses = observed.loc[
    (~observed["keyword_hit_at_5"]) | (~observed["tfidf_hit_at_5"]),
    ["question_id", "keyword_hit_at_5", "tfidf_hit_at_5", "error_category", "qualitative_observation"],
]
display(misses)
print(f"Questions missed by at least one retriever: {len(misses)}")
print(f"Questions missed by both retrievers: {((~observed['keyword_hit_at_5']) & (~observed['tfidf_hit_at_5'])).sum()}")

## 4. How I reproduce the retrieval test

The full incident CSV stays local. With that file present, I can run:

```powershell
python -m scripts.evaluate_incident_assistant --questions evals/incident_eval_questions_provisional.csv
```

This saves detailed local results under ignored `outputs/`. A checked, non-sensitive export is in [reports/retrieval_full_details.csv](../reports/retrieval_full_details.csv). Rerunning with a newer publisher CSV may change the scores.

In [ ]:
# Detailed local retrieval output is optional and deliberately git-ignored.
# If it is present, show a few rows. If absent, keep the committed summary.

raw_result_path = ROOT / "outputs/project3/retrieval_evaluation.csv"
if raw_result_path.exists():
    raw_results = pd.read_csv(raw_result_path, keep_default_na=False)
    display(raw_results[[
        "question", "approach", "expected_doc_id", "retrieved_ids",
        "hit_at_k", "latency_seconds"
    ]].head(10))
    print("Full, reproducible retrieval rows:", len(raw_results))
else:
    print("Full local per-query output not found in this checkout.")
    print("The committed summary and question set above remain visible in GitHub.")

## 5. Evaluate the raw Llama answers separately

I recorded **15 model responses** using retrieved incidents and official-reference cards. For each one, I looked at whether it answered the question, cited the expected incident and refused unavailable information.

The original responses are kept in a local ignored CSV. The following safe summary is committed to GitHub. Its preliminary labels are **not my final human grades**.

To reproduce the local run (Ollama must be running):

```powershell
python -u -m scripts.evaluate_incident_llm --model llama3.2 --base-url http://localhost:11434/v1 --with-knowledge --questions evals/incident_eval_questions_provisional.csv --out outputs/project3/llm_review_first_run.csv
```

Do not count empty manual-review columns as passes. The reviewed guidance shown by Scout's newer UI needs a **separate** evaluation.

In [ ]:
# The full Llama response CSV is stored locally so raw answers can be checked.
# This cell does not automatically approve the human grading columns.
# Missing review values remain missing, never silently counted as passes.

review_path = ROOT / "outputs/project3/llm_review_first_run.csv"
if not review_path.exists():
    print("The 15-answer LLM evaluation has not finished or is not in this workspace.")
    print("No human-approved score is claimed here; the committed AI-assisted provisional evaluation appears below.")
else:
    llm = pd.read_csv(review_path, keep_default_na=False)
    print("Answers recorded:", len(llm))
    print("Total measured generation time (s):", round(pd.to_numeric(llm["latency_seconds"]).sum(), 2))
    print("Model:", "llama3.2 (expected locally — confirm run configuration)")
    display(llm[["question_id", "question", "answer", "cited_ids",
                 "manual_correct", "manual_citations_valid", "manual_refusal_ok",
                 "failure_type", "review_notes"]])
    for col in ["manual_correct", "manual_citations_valid", "manual_refusal_ok"]:
        graded = llm[col].astype(str).str.strip().str.lower()
        assessed = graded.isin(["true", "false", "yes", "no", "1", "0"])
        if assessed.any():
            passed = graded[assessed].isin(["true", "yes", "1"]).sum()
            print(f"{col}: {passed}/{assessed.sum()} assessed — check rubric and relevant subset.")
        else:
            print(f"{col}: NOT GRADED")

## 6. What the first 15 answers showed — provisional

I reviewed the original answers against the expected records using a **strict** rule: a vague reference to something similar is not a correct direct answer.

| Question group | Observed provisional result |
|---|---:|
| Target incident supplied to Llama | **8/12** |
| Factual answer correct | **4/12** |
| Expected incident ID cited | **3/12** |
| Both correct and cited | **2/12** |
| Clearly refused unavailable facts | **2/3** |

These are **assistant-assisted preliminary grades**, not independently signed-off accuracy claims. Correct retrieval, correct answering and correct citation are different checks.

![Raw Llama evidence versus answer quality](../reports/llm_provisional_findings.svg)

In [ ]:
# Calculate provisional rates from the score-only evidence file.
# The source being retrieved, cited and answered correctly are SEPARATE checks.
# All AI-assisted labels remain provisional until independent human review.

# I keep this score-only sheet so the results can be inspected without
# sharing my local Ollama output or SQLite cases.
review = pd.read_csv(ROOT / "reports/llm_provisional_review.csv", keep_default_na=False)
assert len(review) == 15

answerable_review = review[review["answerable"].astype(str).str.lower().eq("true")]
refusal_review = review[review["answerable"].astype(str).str.lower().eq("false")]

retrieval_hit_count = answerable_review["expected_in_top_5"].astype(str).str.lower().eq("true").sum()
correct_count = answerable_review["assistant_provisional_core_answer_pass"].astype(str).str.lower().eq("true").sum()
source_cited_count = answerable_review["expected_doc_cited"].astype(str).str.lower().eq("true").sum()
refusal_count = refusal_review["assistant_provisional_clear_refusal_pass"].astype(str).str.lower().eq("true").sum()
correct_and_cited_count = (
    answerable_review["assistant_provisional_core_answer_pass"].astype(str).str.lower().eq("true")
    & answerable_review["expected_doc_cited"].astype(str).str.lower().eq("true")
).sum()

metrics = pd.DataFrame([
    {"metric": "Target incident retrieved", "passed": int(retrieval_hit_count), "total": 12},
    {"metric": "Correct factual answer (strict, provisional)", "passed": int(correct_count), "total": 12},
    {"metric": "Expected incident explicitly cited", "passed": int(source_cited_count), "total": 12},
    {"metric": "Correct answer AND expected incident cited", "passed": int(correct_and_cited_count), "total": 12},
    {"metric": "Clear refusal on unanswerable questions", "passed": int(refusal_count), "total": 3},
])
metrics["rate"] = metrics["passed"] / metrics["total"]
display(metrics.style.format({"rate": "{:.1%}"}))
assert [int(x) for x in metrics["passed"]] == [8, 4, 3, 2, 2]
print("PROVISIONAL assistant review — NOT independently human approved.")

In [ ]:
# Plot provisional retrieval, answer and citation rates using one denominator
# for the 12 factual questions; refusals have a separate 3-question denominator.

fig, ax = plt.subplots(figsize=(9, 4.5))
view = metrics.iloc[:4].copy()
ax.barh(view["metric"], view["rate"] * 100)
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("Percentage of 12 factual questions")
ax.set_title("Raw Llama 3.2: source availability vs answer correctness")
for y, value in enumerate(view["rate"] * 100):
    ax.text(value + 1, y, f"{value:.1f}%", va="center")
plt.tight_layout()
plt.show()

# A retrieved source is NOT proof the generated answer is correct.
display(review[["question_id", "question_topic", "expected_in_top_5",
                "assistant_provisional_core_answer_pass",
                "assistant_provisional_clear_refusal_pass",
                "failure_category"]])

### What failed, and why

**Azure Q02 and HeyGen Q06:** their correct incidents ranked **first**, but Llama still said no direct match existed. This is a **generation failure**, not a search miss.

**1Password Q01:** it cited the right ID while denying the match. **Circle Q03 and Squarespace Q10:** gave the right fact without the expected source citation.

**Q15:** avoided inventing a revenue figure but failed to say clearly that public reports cannot establish it. Some responses added technical suggestions unrelated to the retrieved report.

These mistakes are why I separated checked instructions from optional model commentary. [Full question-by-question review](../reports/llm_evaluation_findings.md).

In [ ]:
# Convert recorded latency text to numbers before calculating median/mean.
# Report Q15's extreme recorded delay openly; don't use the overall mean to
# describe a typical answer when one observation dominates the total.

durations = pd.to_numeric(review["latency_seconds"])
print("Answers recorded:", len(review))
print("Median raw LLM latency: {:.1f}s".format(durations.median()))
print("Total recorded model time: {:.1f}s".format(durations.sum()))
print("Q15 recorded latency: {:.1f}s".format(
    float(review.loc[review["question_id"] == "Q15", "latency_seconds"].iloc[0])
))
print("Mean excluding anomalous Q15: {:.1f}s".format(
    durations[review["question_id"] != "Q15"].mean()
))
# The safe score-only CSV deliberately excludes raw token-use fields.
# Read exact totals from the locally ignored original answer file, if present.
if review_path.exists():
    original_llm = pd.read_csv(review_path, keep_default_na=False)
    print("Input tokens recorded:", int(pd.to_numeric(original_llm["prompt_tokens"]).sum()))
    print("Output tokens recorded:", int(pd.to_numeric(original_llm["completion_tokens"]).sum()))
else:
    print("Per-answer token totals are retained only in the local raw LLM run file.")

fig, ax = plt.subplots(figsize=(9, 4))
normal = review[review["question_id"] != "Q15"]
ax.bar(normal["question_id"], pd.to_numeric(normal["latency_seconds"]))
ax.set_ylabel("Generation time (seconds)")
ax.set_title("First 14 Llama responses — anomalous Q15 reported separately")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

The 15-answer run recorded a **median of 81.0 seconds**. Its total is **53,453.8 seconds**, but **Q15 alone recorded 52,047.6 seconds**. I cannot establish why, so I show that outlier rather than calling it typical performance.

**Cost:** the recorded run used **19,141 prompt tokens** and **4,314 generated tokens**. Local Ollama incurred **€0 in measured external API fees per question**, but it still used laptop computing time and electricity, which I did not price. Individual question timings are in the saved review CSV; the optional raw token counts remain in the ignored original CSV.

## 7. Why was local Llama slow?

I made **one short direct Ollama request**, separate from the full RAG evaluation. It ran on **100% CPU** and took **22.49 seconds**: 11.54s loading, 1.47s reading the prompt and 9.44s generating 67 tokens (**7.1 tokens/s**).

The diagnostic had 36 input tokens and an 80-token generation limit. This is **one measured hardware check**, not another 15-question benchmark. I cannot assume the earlier model run used the same exact version or hardware state.

Recorded details: [reports/ollama_cpu_single_run_2026-11-10.csv](../reports/ollama_cpu_single_run_2026-11-10.csv).

In [ ]:
# Direct Ollama timing comes from ONE short diagnostic request, NOT 15 trials.
# Compare loading, prompt-processing and generation inside this same request.

# Source: the actual terminal readings, manually recorded in a small,
# non-sensitive CSV. This code reproduces the analysis.
benchmark = pd.read_csv(ROOT / "reports/ollama_cpu_single_run_2026-11-10.csv")
assert len(benchmark) == 1, "Only one directly measured Ollama run has been recorded."
measured = benchmark.iloc[0]

breakdown = pd.DataFrame([
    {"stage": "Load model", "seconds": measured["model_load_seconds"]},
    {"stage": "Process prompt", "seconds": measured["prompt_processing_seconds"]},
    {"stage": "Generate response", "seconds": measured["generation_seconds"]},
])
breakdown["share_of_full_request"] = breakdown["seconds"] / measured["total_seconds"]
display(breakdown.style.format({
    "seconds": "{:.2f}",
    "share_of_full_request": "{:.1%}",
}))

print("Ollama processor:", measured["processor_reported"])
print("Memory (GB):", measured["ram_gb"])
print("Model:", measured["model"])
print("Generated tokens:", int(measured["generated_tokens"]))
print("Output speed: {:.1f} tokens/s".format(measured["output_tokens_per_second"]))
print("Total direct request: {:.2f}s".format(measured["total_seconds"]))
print("Earlier LLM evaluation median: {:.1f}s".format(durations.median()))
print("NOTE: direct Ollama timing and whole-application timing are not interchangeable.")

In [ ]:
# One bar per stage of that single measured direct Ollama request.
# This chart describes where time was spent; it is not a GPU benchmark.

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(breakdown["stage"], breakdown["seconds"])
ax.invert_yaxis()
ax.set_xlabel("Measured seconds in a single direct Ollama request")
ax.set_title("Where local CPU inference time went")
ax.set_xlim(0, 15)
for i, sec in enumerate(breakdown["seconds"]):
    ax.text(sec + 0.2, i, f"{sec:.2f}s", va="center")
plt.tight_layout()
plt.show()

# Illustration ONLY: if output generation remains at this speed,
# a 300-token reply takes about 42 seconds to generate, plus other work.
# This is an extrapolation, not another observed benchmark.
estimated_300_tokens = 300 / float(measured["output_tokens_per_second"])
print("Illustrative generation time for 300 tokens: {:.1f}s (not measured)".format(
    estimated_300_tokens
))

### What I can and can't compare

The **81-second** median describes longer raw Llama answers with retrieved evidence. The separate **22.49-second** direct test used a short question and bypassed Scout. Neither is a measured end-to-end time for the later staged Streamlit interface.

That explains the decision to avoid waiting for generation, **not** a claim that I made Llama compute faster. I would need matched repeat runs to compare models or prompts fairly.

### What I changed because of these results

Scout now **shows and saves the retrieved evidence and checked plan first**. Llama interpretation is a separate choice, with shorter output limits (**200 tokens with checked guidance; 250 otherwise**) and its own unverified label.

I have software tests for the saved reply and model fallback, but **no controlled new UI timing study** or independently scored accuracy test of this staged flow.

**Next honest experiment:** use five new scenarios, measure time to the first checked plan, time and tokens for optional AI, the relevance of the chosen steps, and whether sources actually support each claim.

## 8. My decision

The keyword baseline had **one more retrieval hit** than TF-IDF. I kept TF-IDF for its query speed and documented supplier and wording misses rather than pretend it was more accurate.

Llama sometimes ignored a **correctly retrieved source** and gave unsupported technical suggestions. I therefore made **checked read-only guidance** authoritative for the next step, with model text kept separately.

**Would I ship it?** As a local **investigation aid** with a person checking the evidence, yes for testing. Not as an autonomous root-cause system or secure shared ticket platform.

I still need independent source/answer review, a larger unseen question set and a measured test of the revised UI.

## 9. Evidence and final limitations

I checked that the 12 target records appear in the SevAnalysis collection. I could directly open the original **HeyGen Q06** and **v0 Q12** vendor pages, but **not all 12** original publisher pages. This does not mean the other links are invalid. [Source-check log](../reports/source_link_checks_2026-11-10.md).

The raw responses showed three types of failure: **retrieval missed the target**, **Llama ignored or contradicted a retrieved target**, and **a correct fact had no proper citation**. The provisional **4/12** factual pass rate is not a verified diagnosis rate.

### Evaluation Q&A

**Q. What does Hit@5 tell me?**  
A. Whether the expected incident ID appears in the first five results; not whether Llama answered correctly.

**Q. Why didn't TF-IDF win?**  
A. On these 12 questions it missed more supplier-specific and paraphrased descriptions. It was faster after indexing, which is why I kept it as a limited prototype choice.

**Q. What does RAG add?**  
A. It gives the model retrieved reference material. It doesn't force the model to use that material correctly.

**Q. What about incorrect advice?**  
A. Scout shows checked procedures first, marks optional AI interpretation unverified and asks a human to record what was actually checked. None of that guarantees an accurate diagnosis.

**Q. Is the evaluation final?**  
A. No. The question set was retrospective and the source pages and provisional answer grades need independent human sign-off.